<a id="transformer-overview"></a>

$$
\text{Attention Is All You Need (Vaswani et al., 2017): Transformer} \\ \quad \\
\text{输入: n 个词 (token), 每个词是一个 } d_{model} \text{ 维的向量, 堆成矩阵 } X \in \mathbb{R}^{n \times d_{model}} \\
\text{RNN 必须按顺序一个一个处理 (第 t 步依赖第 t-1 步), 不能并行, 而且远距离的信息要传很多步} \\
\text{Transformer 只用注意力 (Attention): 每个词直接和所有词计算相关性, 一步就能看到全局, 并且可以用矩阵乘法并行} \\ \quad \\
\text{这个 note 用 numpy 从零实现一个编码器层 (Encoder Layer):} \\
\text{缩放点积注意力 → 掩码 → 多头注意力 → 位置编码 → 层归一化与残差 → 前馈网络 → 完整的编码器层}
$$

<a id="scaled-dot-product"></a>

$$
\text{缩放点积注意力 (Scaled Dot-Product Attention)} \\
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V \\ \quad \\
\text{Q (Query 查询), K (Key 键), V (Value 值) 都由输入线性变换得到:} \\
Q = X W_Q \ (n \times d_k), \quad K = X W_K \ (n \times d_k), \quad V = X W_V \ (n \times d_v) \\ \quad \\
\text{逐行理解 (可以看作 "软" 的字典查询):} \\
1. \ \text{第 i 个词用 } q_i \text{ 去和每个词的 } k_j \text{ 比较相似度 (点积): } s_{ij} = \frac{q_i^T k_j}{\sqrt{d_k}} \quad (S = \frac{Q K^T}{\sqrt{d_k}} \text{ 是 } n \times n \text{ 的矩阵}) \\
2. \ \text{对每一行做 softmax 得到权重: } \alpha_{ij} = \frac{e^{s_{ij}}}{\sum_{j'} e^{s_{ij'}}} \text{, 每行非负且和为 1} \\
3. \ \text{输出是所有 value 的加权平均: } o_i = \sum_j \alpha_{ij} v_j \\
\text{普通字典: 找到唯一匹配的 key, 返回它的 value (权重是 one-hot); 注意力: 按相似度给所有 value 分配权重} \\ \quad \\
\text{计算量: } Q K^T \text{ 需要 } O(n^2 d_k) \text{, 序列长度的平方, 这是长文本的瓶颈}
$$

数学基础: [Softmax](softmax_regression.ipynb#softmax) · [矩阵乘法](../math_basics/linear_algebra.ipynb#matrix-multiplication)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax as scipy_softmax

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)       # 数值稳定
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)          # (..., n, n)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)                # mask 为 False 的位置不能看
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

rng = np.random.default_rng(0)
n, d_model, d_k, d_v = 5, 8, 4, 4
X = rng.normal(size=(n, d_model))
W_Q, W_K, W_V = (rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, d)) for d in (d_k, d_k, d_v))
Q, K, V = X @ W_Q, X @ W_K, X @ W_V

out, weights = scaled_dot_product_attention(Q, K, V)
print(f'Q, K, V 形状 = {Q.shape}, {K.shape}, {V.shape}; 注意力权重 {weights.shape}, 输出 {out.shape}')
print(f'每行权重之和 = {weights.sum(axis=1)}, softmax == scipy? {np.allclose(weights, scipy_softmax(Q @ K.T / np.sqrt(d_k), axis=1))}')

# 逐行按定义计算: o_i = sum_j alpha_ij v_j
out_loop = np.zeros_like(out)
for i in range(n):
    s = np.array([Q[i] @ K[j] for j in range(n)]) / np.sqrt(d_k)
    alpha = np.exp(s) / np.exp(s).sum()
    out_loop[i] = sum(alpha[j] * V[j] for j in range(n))
print(f'矩阵形式 == 逐行定义? {np.allclose(out, out_loop)}')

plt.figure(figsize=(4, 3))
plt.imshow(weights, cmap='viridis')
plt.colorbar(label='attention weight')
plt.xlabel('key j')
plt.ylabel('query i')
plt.title('softmax(QK^T / sqrt(d_k))')
plt.tight_layout()
plt.show()

<a id="scaling"></a>

$$
\text{为什么要除以 } \sqrt{d_k} \\ \quad \\
\text{假设 q 和 k 的各分量独立, 均值为 0, 方差为 1, 点积 } q^T k = \sum_{m=1}^{d_k} q_m k_m \\
E[q_m k_m] = E[q_m] E[k_m] = 0 \\
Var(q_m k_m) = E[q_m^2 k_m^2] - (E[q_m k_m])^2 = E[q_m^2] E[k_m^2] - 0 = 1 \times 1 = 1 \\
Var(q^T k) = \sum_{m=1}^{d_k} Var(q_m k_m) = d_k \quad (\text{独立变量的方差可加}) \\
\text{所以点积的标准差是 } \sqrt{d_k} \text{, 除以 } \sqrt{d_k} \text{ 后方差变回 1} \\ \quad \\
\text{不缩放会怎样: } d_k \text{ 很大时 (论文里 } d_k = 64 \text{), 打分的差距很大, softmax 接近 one-hot (饱和)} \\
\text{softmax 的 Jacobian } \text{diag}(p) - p p^T \text{, 当 p 接近 one-hot 时 (例如 } p = e_1 \text{): diag}(e_1) - e_1 e_1^T = 0 \\
\text{梯度几乎为 0, 参数学不动 (梯度消失)}
$$

数学基础: [期望和方差的性质](../math_basics/statistics.ipynb#expectation-variance-properties) · [Softmax 的 Jacobian](softmax_regression.ipynb#softmax-jacobian)

In [ ]:
rng = np.random.default_rng(1)
print('点积的方差 (100000 对随机向量):')
for dk in [4, 16, 64, 256]:
    q = rng.normal(size=(100000, dk))
    k = rng.normal(size=(100000, dk))
    dots = np.sum(q * k, axis=1)
    print(f'  d_k = {dk:>3}: Var(q.k) = {dots.var():8.3f} (理论 {dk}), 除以 sqrt(d_k) 后 Var = {(dots / np.sqrt(dk)).var():.3f}')

# softmax 饱和: 一行 10 个打分, 看最大权重和 Jacobian 的大小
def softmax_jacobian_norm(s):
    p = softmax(s)
    return np.linalg.norm(np.diag(p) - np.outer(p, p))

dks = [4, 16, 64, 256, 1024]
max_w_raw, max_w_scaled, jac_raw, jac_scaled = [], [], [], []
for dk in dks:
    q = rng.normal(size=(500, 1, dk))
    k = rng.normal(size=(500, 10, dk))
    s = np.sum(q * k, axis=2)                      # 500 行, 每行 10 个打分
    for s_use, mw, jn in [(s, max_w_raw, jac_raw), (s / np.sqrt(dk), max_w_scaled, jac_scaled)]:
        mw.append(np.mean(softmax(s_use).max(axis=1)))
        jn.append(np.mean([softmax_jacobian_norm(row) for row in s_use]))
for dk, a, b, c, d in zip(dks, max_w_raw, max_w_scaled, jac_raw, jac_scaled):
    print(f'd_k = {dk:>4}: 平均最大权重 不缩放 = {a:.3f}, 缩放 = {b:.3f}; Jacobian 范数 不缩放 = {c:.4f}, 缩放 = {d:.4f}')

plt.figure(figsize=(4, 3))
plt.semilogx(dks, jac_raw, 'o-', label='QK^T (no scaling)')
plt.semilogx(dks, jac_scaled, 's-', label='QK^T / sqrt(d_k)')
plt.xlabel('d_k')
plt.ylabel('|softmax Jacobian|')
plot_nicely()
plt.show()

<a id="masking"></a>

$$
\text{掩码 (Masking)} \\
\text{解码器生成第 i 个词时, 不能看到后面还没生成的词 (因果掩码 Causal Mask):} \\
s_{ij} = -\infty \quad (j > i) \implies \alpha_{ij} = \frac{e^{-\infty}}{\sum} = 0 \\
\text{掩码矩阵是下三角的: 第 i 行只有前 i 个位置可以看} \\
\text{(另一种是填充掩码 Padding Mask: 一个 batch 里句子长短不同, 补齐的位置不能被注意)} \\ \quad \\
\text{验证方法: 修改第 j 个词之后的输入, 前 j 个位置的输出应该完全不变}
$$

In [ ]:
causal_mask = np.tril(np.ones((n, n), dtype=bool))
out_c, w_c = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f'因果掩码的权重矩阵 (上三角全是 0): \n{np.round(w_c, 3)}')

# 修改最后两个词, 前 3 个位置的输出不变
X_changed = X.copy()
X_changed[3:] = rng.normal(size=(2, d_model))
out_c2, _ = scaled_dot_product_attention(X_changed @ W_Q, X_changed @ W_K, X_changed @ W_V, mask=causal_mask)
print(f'修改第 4, 5 个词后: 前 3 个位置输出不变? {np.allclose(out_c[:3], out_c2[:3])}, 后 2 个位置改变? {not np.allclose(out_c[3:], out_c2[3:])}')

<a id="multi-head"></a>

$$
\text{多头注意力 (Multi-Head Attention)} \\
\text{一组 } W_Q, W_K, W_V \text{ 只能学到一种 "相关性", 用 h 组并行, 每组 (每个头) 关注不同的模式} \\
\text{head}_i = \text{Attention}(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)}), \quad W_Q^{(i)}, W_K^{(i)} \in \mathbb{R}^{d_{model} \times d_k}, \ W_V^{(i)} \in \mathbb{R}^{d_{model} \times d_v} \\
\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \cdots, \text{head}_h) \, W_O, \quad W_O \in \mathbb{R}^{h d_v \times d_{model}} \\ \quad \\
\text{取 } d_k = d_v = d_{model} / h \text{ (论文: } d_{model} = 512, h = 8, d_k = 64 \text{), 参数量:} \\
W_Q, W_K, W_V \text{ 各 h 个 } d_{model} \times \frac{d_{model}}{h} \text{ 合起来各 } d_{model}^2 \text{, 加上 } W_O \text{ 的 } d_{model}^2 \text{, 共 } 4 d_{model}^2 \\
\text{和一个 } d_{model} \text{ 维的单头注意力一样多, 计算量也相同} \\ \quad \\
\text{实现技巧: 把 h 个 } W_Q^{(i)} \text{ 横着拼成一个 } d_{model} \times d_{model} \text{ 的矩阵, 一次乘完, 再 reshape 成 (h, n, d_k) 批量计算}
$$

In [ ]:
def multi_head_attention(X, W_Q, W_K, W_V, W_O, h, mask=None):
    n_tok, dm = X.shape
    dk = dm // h
    def split_heads(M):                     # (n, d_model) -> (h, n, d_k)
        return M.reshape(n_tok, h, dk).transpose(1, 0, 2)
    Qh, Kh, Vh = split_heads(X @ W_Q), split_heads(X @ W_K), split_heads(X @ W_V)
    heads, w = scaled_dot_product_attention(Qh, Kh, Vh, mask)          # (h, n, d_k)
    concat = heads.transpose(1, 0, 2).reshape(n_tok, dm)               # (n, h * d_k)
    return concat @ W_O, w

d_model, h, n = 16, 4, 6
rng = np.random.default_rng(2)
X = rng.normal(size=(n, d_model))
W_Q, W_K, W_V, W_O = (rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, d_model)) for _ in range(4))
mha_out, mha_w = multi_head_attention(X, W_Q, W_K, W_V, W_O, h)

# 按定义逐个头计算: 第 i 个头用 W_Q 的第 i 块列
dk = d_model // h
heads = []
for i in range(h):
    cols = slice(i * dk, (i + 1) * dk)
    head_i, _ = scaled_dot_product_attention(X @ W_Q[:, cols], X @ W_K[:, cols], X @ W_V[:, cols])
    heads.append(head_i)
loop_out = np.concatenate(heads, axis=1) @ W_O
print(f'输出形状 = {mha_out.shape}, 每个头的权重形状 = {mha_w.shape}')
print(f'reshape 批量计算 == 逐个头计算再拼接? {np.allclose(mha_out, loop_out)}')
print(f'参数量 = {W_Q.size + W_K.size + W_V.size + W_O.size} = 4 * d_model^2 = {4 * d_model**2}')

fig, axs = plt.subplots(1, h, figsize=(8, 2.4))
for i in range(h):
    axs[i].imshow(mha_w[i], cmap='viridis', vmin=0, vmax=1)
    axs[i].set_title(f'head {i + 1}')
    axs[i].set_xticks([]); axs[i].set_yticks([])
plt.tight_layout()
plt.show()

<a id="positional-encoding"></a>

$$
\text{位置编码 (Positional Encoding)} \\ \quad \\
\text{1. 为什么需要: 注意力本身不知道词的顺序 (置换等变 Permutation Equivariant)} \\
\text{把输入的行打乱: } X' = P X \text{ (P 是置换矩阵, } P^T P = I \text{), 则 } Q' = P Q, K' = P K, V' = P V \\
\text{softmax}\left(\frac{P Q K^T P^T}{\sqrt{d_k}}\right) P V = P \, \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) P^T P V = P \cdot \text{Attention}(Q, K, V) \\
\text{(按行 softmax 时, 行和列同时置换只是把结果的行列同样置换)} \\
\text{输出只是跟着输入一起被打乱, "我爱你" 和 "你爱我" 每个词得到的表示完全一样, 所以必须把位置信息加进输入} \\ \quad \\
\text{2. 正弦位置编码: } \omega_i = \frac{1}{10000^{2i / d_{model}}}, \ i = 0, 1, \cdots, \frac{d_{model}}{2} - 1 \\
PE_{(pos, 2i)} = \sin(pos \cdot \omega_i), \quad PE_{(pos, 2i+1)} = \cos(pos \cdot \omega_i) \\
\text{输入变成 } X + PE \text{. 不同维度的频率从 1 到 } \frac{1}{10000} \text{, 像一个多进制的 "时钟"} \\ \quad \\
\text{3. 相对位置是线性变换 (旋转): 由和角公式} \\
\sin((pos + k)\omega) = \sin(pos\,\omega)\cos(k\omega) + \cos(pos\,\omega)\sin(k\omega) \\
\cos((pos + k)\omega) = \cos(pos\,\omega)\cos(k\omega) - \sin(pos\,\omega)\sin(k\omega) \\
\begin{bmatrix} \sin((pos + k)\omega) \\ \cos((pos + k)\omega) \end{bmatrix} = \underbrace{\begin{bmatrix} \cos k\omega & \sin k\omega \\ -\sin k\omega & \cos k\omega \end{bmatrix}}_{R(k\omega)} \begin{bmatrix} \sin(pos\,\omega) \\ \cos(pos\,\omega) \end{bmatrix} \\
R \text{ 只和偏移 k 有关, 和 pos 无关; 并且 } R^T R = I \text{ (正交矩阵, 旋转)} \\
\text{所以 } PE_{pos + k} = M_k \, PE_{pos} \text{ (} M_k \text{ 是每两维一个旋转块的分块对角矩阵), 模型容易学到相对位置} \\ \quad \\
\text{4. 推论: 两个位置编码的点积只和距离有关} \\
PE_{pos}^T PE_{pos + k} = \sum_i [\sin(pos\,\omega_i)\sin((pos+k)\omega_i) + \cos(pos\,\omega_i)\cos((pos+k)\omega_i)] = \sum_i \cos(k \omega_i)
$$

数学基础: [正交矩阵与旋转](../math_basics/linear_algebra.ipynb#orthogonal) · [欧拉公式 (旋转 = 乘以 e^{ikω})](../math_basics/calculus.ipynb#taylor)

In [ ]:
def positional_encoding(n_pos, d_model):
    pos = np.arange(n_pos)[:, None]
    omega = 1 / 10000 ** (2 * np.arange(d_model // 2) / d_model)
    PE = np.zeros((n_pos, d_model))
    PE[:, 0::2] = np.sin(pos * omega)
    PE[:, 1::2] = np.cos(pos * omega)
    return PE, omega

# 1. 置换等变: 没有位置编码时, 打乱输入只会打乱输出
perm = rng.permutation(n)
out_x, _ = multi_head_attention(X, W_Q, W_K, W_V, W_O, h)
out_px, _ = multi_head_attention(X[perm], W_Q, W_K, W_V, W_O, h)
print(f'Attention(PX) == P Attention(X) ? {np.allclose(out_px, out_x[perm])}')
PE_small, _ = positional_encoding(n, d_model)
out_pe, _ = multi_head_attention(X + PE_small, W_Q, W_K, W_V, W_O, h)
out_ppe, _ = multi_head_attention(X[perm] + PE_small, W_Q, W_K, W_V, W_O, h)
print(f'加上位置编码后还等变吗? {np.allclose(out_ppe, out_pe[perm])} (顺序信息被保留了)')

# 2. 相对位置 = 旋转
PE, omega = positional_encoding(100, 64)
k_shift = 7
M_k = np.zeros((64, 64))
for i, w in enumerate(omega):
    c, s = np.cos(k_shift * w), np.sin(k_shift * w)
    M_k[2 * i:2 * i + 2, 2 * i:2 * i + 2] = [[c, s], [-s, c]]   # 作用在 (sin, cos) 这一对上
print(f'PE[pos + {k_shift}] == M_k PE[pos] 对所有 pos 成立? {np.allclose(PE[k_shift:], PE[:-k_shift] @ M_k.T)}')
print(f'M_k 是正交矩阵? {np.allclose(M_k.T @ M_k, np.eye(64))}')

# 3. 点积只和距离有关
dots_k = [PE[p] @ PE[p + k_shift] for p in range(0, 90, 10)]
print(f'不同 pos 的 PE[pos] . PE[pos + {k_shift}] = {np.round(dots_k, 6)}, 公式 sum cos(k w) = {np.sum(np.cos(k_shift * omega)):.6f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
im = axs[0].imshow(PE, aspect='auto', cmap='RdBu')
axs[0].set_xlabel('dimension')
axs[0].set_ylabel('position')
plt.colorbar(im, ax=axs[0])
axs[1].plot([PE[0] @ PE[k] for k in range(100)], label='PE[0] . PE[k]')
axs[1].set_xlabel('distance k')
plot_nicely()
plt.show()

<a id="layer-norm"></a>

$$
\text{层归一化 (Layer Normalization) 与残差连接 (Residual Connection)} \\ \quad \\
\text{层归一化: 对每个词 (每一行) 单独在特征维度上标准化, 再做一个可学习的缩放和平移} \\
\mu = \frac{1}{d} \sum_{m=1}^d x_m, \quad \sigma^2 = \frac{1}{d} \sum_{m=1}^d (x_m - \mu)^2, \quad \text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta \\
\text{(批归一化 BatchNorm 是在 batch 维度上对每个特征标准化, 依赖 batch 大小; LayerNorm 每个样本独立, 适合变长序列)} \\ \quad \\
\text{残差连接: 子层的输出加上输入 } y = x + F(x) \\
\frac{\partial y}{\partial x} = I + \frac{\partial F}{\partial x} \quad (\text{Jacobian 多了一个单位矩阵}) \\
\text{很多层叠在一起时, 梯度是很多个 Jacobian 相乘; 有了 I, 即使 } \frac{\partial F}{\partial x} \text{ 很小, 梯度也能直接沿着 "捷径" 传回去, 不会消失} \\ \quad \\
\text{论文的编码器子层: } \text{LayerNorm}(x + \text{Sublayer}(x))
$$

数学基础: [标准化 (Z-Score)](../math_basics/statistics.ipynb#standardization) · [Jacobian 与链式法则](../math_basics/calculus.ipynb#jacobian)

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)          # 除以 d (ddof = 0)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

x_ln = rng.normal(5, 3, size=(n, d_model))
ln_out = layer_norm(x_ln, np.ones(d_model), np.zeros(d_model))
print(f'LayerNorm 后每一行: 均值 = {np.round(ln_out.mean(axis=1), 6) + 0.0}, 标准差 = {np.round(ln_out.std(axis=1), 4)}')
# 和逐行用 z-score 公式比较
zs = np.array([(row - row.mean()) / np.sqrt(row.var() + 1e-5) for row in x_ln])
print(f'== 逐行 z-score ? {np.allclose(ln_out, zs)}')

# 残差连接的 Jacobian = I + dF/dx (用一个很小的线性子层 F(x) = x A)
A_small = 0.01 * rng.normal(size=(d_model, d_model))
J_res = np.eye(d_model) + A_small.T
print(f'20 层 F 的 Jacobian 连乘的范数 = {np.linalg.norm(np.linalg.matrix_power(A_small.T, 20)):.2e}, '
      f'20 层残差块连乘的范数 = {np.linalg.norm(np.linalg.matrix_power(J_res, 20)):.2f} (梯度没有消失)')

<a id="encoder-block"></a>

$$
\text{完整的编码器层 (Encoder Layer)} \\
\text{1. 多头自注意力 + 残差 + 层归一化: } H = \text{LN}(X + \text{MultiHead}(X)) \\
\text{2. 逐位置前馈网络 (Position-wise Feed-Forward Network), 每个词用同一个两层 MLP 独立处理:} \\
\text{FFN}(h) = \max(0, h W_1 + b_1) W_2 + b_2, \quad W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}, \ W_2 \in \mathbb{R}^{d_{ff} \times d_{model}} \quad (\text{论文 } d_{ff} = 2048 = 4 d_{model}) \\
\text{3. 再一次残差 + 层归一化: } \text{Out} = \text{LN}(H + \text{FFN}(H)) \\ \quad \\
\text{输入输出形状都是 } (n, d_{model}) \text{, 所以可以把 N 层叠起来 (论文 N = 6)} \\
\text{注意力负责词与词之间交换信息, FFN 负责对每个词单独做非线性变换}
$$

数学基础: [神经网络的前向传播](neural_network.ipynb#mlp-forward) · [Xavier 初始化](neural_network.ipynb#initialization)

In [ ]:
def init_encoder_layer(d_model, d_ff, rng):
    s = lambda fan_in: 1 / np.sqrt(fan_in)
    return {
        'W_Q': rng.normal(0, s(d_model), (d_model, d_model)), 'W_K': rng.normal(0, s(d_model), (d_model, d_model)),
        'W_V': rng.normal(0, s(d_model), (d_model, d_model)), 'W_O': rng.normal(0, s(d_model), (d_model, d_model)),
        'W1': rng.normal(0, s(d_model), (d_model, d_ff)), 'b1': np.zeros(d_ff),
        'W2': rng.normal(0, s(d_ff), (d_ff, d_model)), 'b2': np.zeros(d_model),
        'g1': np.ones(d_model), 'be1': np.zeros(d_model), 'g2': np.ones(d_model), 'be2': np.zeros(d_model),
    }

def encoder_layer(X, p, h, mask=None):
    n_tok, dm = X.shape
    attn, w = multi_head_attention(X, p['W_Q'], p['W_K'], p['W_V'], p['W_O'], h, mask)
    assert attn.shape == (n_tok, dm)
    H = layer_norm(X + attn, p['g1'], p['be1'])                   # 残差 + LN
    ffn = np.maximum(0, H @ p['W1'] + p['b1']) @ p['W2'] + p['b2'] # ReLU 前馈网络
    assert ffn.shape == (n_tok, dm)
    return layer_norm(H + ffn, p['g2'], p['be2']), w

d_model, h, d_ff, n_tok, N_layers = 32, 4, 128, 10, 6
rng = np.random.default_rng(5)
tokens = rng.normal(size=(n_tok, d_model))                        # 假设这是 10 个词的词向量
PE, _ = positional_encoding(n_tok, d_model)
x_enc = tokens + PE
layers = [init_encoder_layer(d_model, d_ff, rng) for _ in range(N_layers)]
for li, p in enumerate(layers):
    x_enc, w = encoder_layer(x_enc, p, h)
    print(f'第 {li + 1} 层: 输出形状 = {x_enc.shape}, 注意力权重形状 = {w.shape}, 每行均值 ≈ {np.abs(x_enc.mean(axis=1)).max():.1e}, 标准差 ≈ {x_enc.std(axis=1).mean():.4f}')

n_params = sum(v.size for v in layers[0].values())
print(f'一层的参数量 = {n_params} = 4 d^2 (注意力) + 2 d d_ff + d_ff + d (FFN) + 4 d (两个 LN) = {4 * d_model**2 + 2 * d_model * d_ff + d_ff + d_model + 4 * d_model}')